In [15]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
import ast
from datetime import timedelta

In [16]:
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=181.57.189.150,1433;"
    "DATABASE=master;"
    "UID=sa;"
    "PWD=P@ssw0rd*;"
)

# Crear el motor de conexión
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [17]:
query_data = """
SELECT DISTINCT 'C' + c.Id AS Cliente,
       c.Nombre,
       c.Email AS Email,
       c.Telefono AS Celular,
       v.Fecha AS Fecha_Venta,
       v.Canal,
       SUM(v.Valor_Neto) AS Venta
FROM Ventas_Shopify.dbo.Contacto_Clientes c
INNER JOIN Ventas_Shopify.dbo.Ventas_ShopifyMoviNova v ON c.id = v.id_cliente
WHERE v.fecha >= '2025-07-17'
    --AND v.Fecha <= '2025-07-15'
    AND Canal != 'MARKETPLACE'
GROUP BY c.Id,
       c.Nombre,
       c.Email,
       c.Telefono,
       v.Fecha,
       v.Canal
"""

# Ejecutar y cargar en DataFrame
ventas_shopify = pd.read_sql(query_data, engine)

query_com = """
SELECT DISTINCT c.CLICodigo AS Cliente,
       c.CLINombres + ' ' + c.CLIApellidos AS Nombre,
       c.CLIEmailPrincipal AS Email,
       c.CLICelular AS Celular,
       v.Fecha AS Fecha_Venta,
       v.Canal,
       SUM(v.Venta_Neta) AS Venta
FROM Ventas_Comerssia.dbo.Contacto_Clientes c
INNER JOIN Ventas_Comerssia.dbo.Ventas_Comerssia v ON c.CLICodigo = v.Cliente
WHERE v.fecha >= '2025-07-17'
    --AND v.Fecha <= '2025-07-15'
    AND Canal != 'RAPPI' AND Canal != 'C&C'
GROUP BY  c.CLICodigo,
       c.CLINombres + ' ' + c.CLIApellidos,
       c.CLIEmailPrincipal,
       c.CLICelular,
       v.Fecha,
       v.Canal
"""

# Ejecutar y cargar en DataFrame
ventas_comerssia = pd.read_sql(query_com, engine)
df_ventas = ventas_comerssia
# df_ventas = pd.concat([ventas_comerssia, ventas_shopify], ignore_index=True)
df_ventas.head()

,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta
0,C005977,ANA SANTOS,anagabisaol@gmail.com,3108878272,2025-07-23,Retail,134453.78
1,C012707940018,TATIANA LEIVA,negado@provenzal.net,1111111111,2025-07-21,Retail,74789.92
2,C0270582,YUMI WAN,negado@provenzal.net,1111111111,2025-07-20,Retail,161428.57
3,C07375790,MONSERRAT CASTANEDA,negado@provenzal.net,1111111111,2025-07-21,Retail,264201.68
4,C08201953,CARMEN MEDINA,negado@provenzal.net,1111111111,2025-07-21,Retail,90756.30


In [18]:
# #consulta segmentada 
# query_data = """
# SELECT DISTINCT 'C' + c.Id AS Cliente,
#        c.Nombre,
#        c.Email AS Email,
#        c.Telefono AS Celular,
#        v.Fecha AS Fecha_Venta,
#        v.Canal,
#        SUM(h.Venta_Neta) AS Venta
# FROM Ventas_Shopify.dbo.Contacto_Clientes c
# INNER JOIN Ventas_Shopify.dbo.Ventas_ShopifyMoviNova v ON c.id = v.id_cliente
# INNER JOIN Ventas_Shopify.dbo.Venta_Homologada h ON v.Pedido = h.Pedido
# WHERE v.fecha >= '2025-07-17'
#     --AND v.Fecha <= '2025-07-15'
#     AND Canal != 'MARKETPLACE'  
#     AND h.Categoria = 'CUIDADO CAPILAR'
#     AND h.Nombre NOT LIKE '%Acond%'
# GROUP BY c.Id,
#        c.Nombre,
#        c.Email,
#        c.Telefono,
#        v.Fecha,
#        v.Canal
# """

# # Ejecutar y cargar en DataFrame
# ventas_shopify = pd.read_sql(query_data, engine)

# # query_com = """
# # SELECT DISTINCT c.CLICodigo AS Cliente,
# #        c.CLINombres + ' ' + c.CLIApellidos AS Nombre,
# #        c.CLIEmailPrincipal AS Email,
# #        c.CLICelular AS Celular,
# #        v.Fecha AS Fecha_Venta,
# #        v.Canal,
# #        SUM(v.Venta_Neta) AS Venta
# # FROM Ventas_Comerssia.dbo.Contacto_Clientes c
# # INNER JOIN Ventas_Comerssia.dbo.Ventas_Comerssia v ON c.CLICodigo = v.Cliente
# # WHERE v.fecha >= '2025-07-17'
# #     --AND v.Fecha <= '2025-07-15'
# #     AND Canal != 'RAPPI' AND Canal != 'C&C'
# #     AND v.Categoria = 3
# #     AND v.Descripcion NOT LIKE '%Acond%'
# # GROUP BY  c.CLICodigo,
# #        c.CLINombres + ' ' + c.CLIApellidos,
# #        c.CLIEmailPrincipal,
# #        c.CLICelular,
# #        v.Fecha,
# #        v.Canal
# # """

# # # Ejecutar y cargar en DataFrame
# # ventas_comerssia = pd.read_sql(query_com, engine)
# df_ventas = ventas_shopify
# # df_ventas = pd.concat([ventas_comerssia, ventas_shopify], ignore_index=True)
# df_ventas.head()

In [19]:
# # CONSULTAS CUMPLEAÑOS

# query_data = """
# SELECT DISTINCT c.Id AS Cliente,
#        c.Nombre,
#        c.Email AS Email,
#        c.Telefono AS Celular,
#        v.Fecha AS Fecha_Venta,
#        v.Canal,
#        SUM(v.Valor_Neto) AS Venta
# FROM Ventas_Shopify.dbo.Contacto_Clientes c
# INNER JOIN Ventas_Shopify.dbo.Ventas_ShopifyMoviNova v ON c.id = v.id_cliente
# WHERE v.fecha >= '2025-06-06'
#     AND v.Fecha <= '2025-06-30'
#     AND v.Codigo_Descuento LIKE '%CUMP%'
# GROUP BY c.Id,
#        c.Nombre,
#        c.Email,
#        c.Telefono,
#        v.Fecha,
#        v.Canal
# """

# # Ejecutar y cargar en DataFrame
# ventas_shopify = pd.read_sql(query_data, engine)

# query_com = """
# SELECT DISTINCT c.CLICodigo AS Cliente,
#        c.CLINombres + ' ' + c.CLIApellidos AS Nombre,
#        c.CLIEmailPrincipal AS Email,
#        c.CLICelular AS Celular,
#        v.Fecha AS Fecha_Venta,
#        v.Canal,
#        SUM(v.Venta_Neta) AS Venta
# FROM Ventas_Comerssia.dbo.Contacto_Clientes c
# INNER JOIN Ventas_Comerssia.dbo.Ventas_Comerssia v ON c.CLICodigo = v.Cliente
# WHERE v.Fecha >= '2025-06-06'
#     AND v.Fecha <= '2025-06-30'
#     AND v.ICPDescripcion LIKE '%CUMP%'
# GROUP BY  c.CLICodigo,
#        c.CLINombres + ' ' + c.CLIApellidos,
#        c.CLIEmailPrincipal,
#        c.CLICelular,
#        v.Fecha,
#        v.Canal
# """

# # Ejecutar y cargar en DataFrame
# ventas_comerssia = pd.read_sql(query_com, engine)

# ventas_total = pd.concat([ventas_comerssia, ventas_shopify], ignore_index=True)
# ventas_total.head()

In [20]:
# Leer archivo
df_indigitall = pd.read_excel("Atribucion.xlsx")

# Seleccionar columnas relevantes
df_campania = df_indigitall[['email', 'campaignName', 'sentAt', 'openedAt', 'clicks']].copy()

# Renombrar columnas
df_campania.rename(columns={
    'email': 'Email',
    'campaignName': 'campania',
    'sentAt': 'fecha_envio',
    'openedAt': 'apertura',
    'clicks': 'clics'
}, inplace=True)

# 1. Convertir la columna openedAt de string a lista real de fechas
df_campania['apertura'] = df_campania['apertura'].astype(str)
df_campania['apertura'] = df_campania['apertura'].apply(
    lambda x: ast.literal_eval(x) if x.startswith("[") else []
)

# 2. Expandir: cada fecha en una fila
df_expanded = df_campania.explode('apertura').copy()

# 3. Limpiar y convertir a datetime
df_expanded['apertura'] = df_expanded['apertura'].str.strip()
df_expanded['apertura'] = pd.to_datetime(
    df_expanded['apertura'],
    format='%Y-%m-%dT%H:%M:%S.%fZ',
    utc=True,
    errors='coerce'
).dt.date

# 4. Normalizar correos
df_expanded['Email'] = df_expanded['Email'].str.lower().str.strip()

# 5. Elegir solo la primera apertura por email
df_abiertos = df_expanded.dropna(subset=['apertura']) \
    .sort_values('apertura') \
    .drop_duplicates(subset='Email', keep='first')

# Filtro correos abiertos hasta una fecha maxima
fecha_maxima = pd.to_datetime('2025-07-31').date()
df_abiertos_filtrados = df_abiertos[df_abiertos['apertura'] <= fecha_maxima]

total_abiertos = df_abiertos_filtrados['Email'].nunique()
print(f"Número de clientes que abrieron hasta {fecha_maxima}: {total_abiertos}")

df_abiertos_filtrados.head()

Número de clientes que abrieron hasta 2025-07-31: 1034


,Email,campania,fecha_envio,apertura,clics
3,lauramoreno.radio@gmail.com,Sorteo Anchetas,2025-07-26T16:01:02.000Z,2025-07-26,[]
7402,marthalucia.covelli@yahoo.com.co,Sorteo Anchetas,2025-07-26T16:00:06.000Z,2025-07-26,[]
7450,claudiaj3331@gmail.com,Sorteo Anchetas,2025-07-26T16:00:58.000Z,2025-07-26,"[{'clicked_at': '2025-07-26T16:13:37.000Z', 'c..."
7480,nora.munar@leadconsultingco.com,Sorteo Anchetas,2025-07-26T16:00:24.000Z,2025-07-26,[]
7481,itamariamartinez@yahoo.com,Sorteo Anchetas,2025-07-26T16:00:38.000Z,2025-07-26,[]


In [21]:
# Normalizar df_ventas
df_ventas['Email'] = df_ventas['Email'].str.lower().str.strip()
df_ventas['Fecha_Venta'] = pd.to_datetime(df_ventas['Fecha_Venta']).dt.date

# Merge por email
df_merge = pd.merge(df_ventas, df_abiertos, on='Email', how='inner')

# Filtrar ventas dentro de 5 días desde la apertura
df_merge['dias_post_apertura'] = (df_merge['Fecha_Venta'] - df_merge['apertura']).apply(lambda x: x.days)
df_atribucion = df_merge[
    (df_merge['dias_post_apertura'] >= 0) &
    (df_merge['dias_post_apertura'] <= 5)
]

# Sumar la venta total atribuida
tottal_venta = df_atribucion['Venta'].sum()

filas = len(df_atribucion) # Ver cuántas filas hay realmente

print(f"Total de filas: {filas}")
print(f"Total de venta atribuida: {tottal_venta:,.0f}")
# Resultado final
df_atribucion.head()

Total de filas: 5
Total de venta atribuida: 1,364,790


,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta,campania,fecha_envio,apertura,clics,dias_post_apertura
1,C1127233163,SANTIAGO LONDONO,slondon0@me.com,3106627,2025-07-27,Retail,153655.46,Sorteo Anchetas,2025-07-26T16:00:36.000Z,2025-07-26,[],1
14,C52931528,JESSICA VILLA,jessivillainfante@icloud.com,3164300870,2025-08-01,Retail,642226.89,Sorteo Anchetas,NaN,2025-07-27,[],5
19,C900087104,SMART PACK PACK,gerencia@smartpackcolombia.com,3165260593,2025-07-27,Retail,208403.37,Sorteo Anchetas,2025-07-26T16:00:33.000Z,2025-07-26,[],1
25,C901114488,None,facturacionaguamariacolombia@gmail.com,3116099817,2025-07-31,Retail,216806.72,Sorteo Anchetas,NaN,2025-07-30,[],1
29,C901226368,None,gerencia@unlimark.com,3112788310,2025-07-27,Retail,143697.48,Sorteo Anchetas,NaN,2025-07-27,[],0
